# Gaussian Naïve Bayes Classifier

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## A Completely Different Approach

Every algorithm so far found a **decision boundary** — a line, a curve, a margin — that separates the classes. Naïve Bayes does something fundamentally different:

> *Instead of finding a boundary, directly estimate the probability that a sample belongs to each class using Bayes' theorem.*

### Bayes' Theorem

$$P(\text{class} \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid \text{class}) \times P(\text{class})}{P(\mathbf{x})}$$

| Term | Name | Meaning |
|---|---|---|
| $P(\text{class} \mid \mathbf{x})$ | Posterior | What we want: P(home win given these features) |
| $P(\mathbf{x} \mid \text{class})$ | Likelihood | How often do these features appear in home wins? |
| $P(\text{class})$ | Prior | How often do home wins occur overall? |
| $P(\mathbf{x})$ | Evidence | Same for all classes — we ignore it |

We compute the posterior for each class and predict the one with the highest value.

### The Naïve Assumption

"Naïve" refers to assuming all features are **conditionally independent** given the class:

$$P(\mathbf{x} \mid \text{class}) = \prod_{j=1}^{n} P(x_j \mid \text{class})$$

This is almost never true — home goals rolling and home win rate are clearly correlated. But the classifier works surprisingly well despite the violation.

### Why Gaussian?

Our features are continuous numbers, not categories. We assume each feature follows a **normal distribution** within each class. Training just means computing the mean and variance of each feature per class — then evaluating how likely a new value is under that Gaussian.

### Training Speed

No gradient descent. No iterations. Training is a single pass through the data to compute means and variances. **Instant.**


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.naive_bayes import GaussianNaiveBayes

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. The Gaussian Assumption — Visualised

For each feature, Naïve Bayes fits a Gaussian to each class. Let's see what that looks like on toy data before touching football.

In [ ]:
rng = np.random.default_rng(0)
X0 = rng.normal(loc=[-2], scale=0.8, size=(100, 1))
X1 = rng.normal(loc=[2],  scale=0.8, size=(100, 1))
X_toy = np.vstack([X0, X1])
y_toy = np.array([0]*100 + [1]*100)

m_toy = GaussianNaiveBayes().fit(X_toy, y_toy)

z = np.linspace(-5, 5, 300)
def gaussian(z, mean, var):
    return np.exp(-0.5 * (z - mean)**2 / var) / np.sqrt(2 * np.pi * var)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(X0, bins=20, alpha=0.4, color='#E87272', density=True, label='Class 0 data')
ax.hist(X1, bins=20, alpha=0.4, color='#4878CF', density=True, label='Class 1 data')
ax.plot(z, gaussian(z, m_toy.means_[0][0], m_toy.variances_[0][0]),
        color='#E87272', linewidth=2.5, label='Fitted Gaussian (class 0)')
ax.plot(z, gaussian(z, m_toy.means_[1][0], m_toy.variances_[1][0]),
        color='#4878CF', linewidth=2.5, label='Fitted Gaussian (class 1)')
ax.set_xlabel('Feature value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Gaussian Naïve Bayes — Fitted Distributions per Class', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

For a new point, Naïve Bayes asks: "how likely is this value under the class-0 Gaussian vs the class-1 Gaussian?" The class whose Gaussian assigns higher probability wins — multiplied by the prior.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Naïve Bayes doesn't technically require scaling — it fits per-feature
# Gaussians independently. But scaling can help numerical stability.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 4. Train — Instant

In [ ]:
import time
start = time.time()
model = GaussianNaiveBayes().fit(X_train_sc, y_train)
elapsed = time.time() - start

print(f'Training time: {elapsed*1000:.2f} ms')
print(f'Train accuracy: {model.score(X_train_sc, y_train):.3f}')
print(f'Test  accuracy: {model.score(X_test_sc,  y_test):.3f}')

No epochs, no gradient descent — just compute statistics from data. Training completes in milliseconds regardless of dataset size.

---
## 5. Learned Distributions per Feature

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()
z = np.linspace(-4, 4, 300)

def gaussian_pdf(z, mean, var):
    return np.exp(-0.5 * (z - mean)**2 / var) / np.sqrt(2 * np.pi * var)

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    for c, color, label in [(0, '#E87272', 'Draw/Away'), (1, '#4878CF', 'Home Win')]:
        mean = model.means_[c][i]
        var  = model.variances_[c][i]
        ax.plot(z, gaussian_pdf(z, mean, var), color=color, linewidth=2, label=label)
        ax.axvline(mean, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

axes[-1].axis('off')
plt.suptitle('Fitted Gaussians per Feature per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Each subplot shows the fitted Gaussian for one feature, split by class. Where the two curves are far apart, that feature is very informative — home wins and non-home-wins have very different distributions for that feature. Where they overlap heavily, the feature carries little information.

---
## 6. Confusion Matrix & ROC

In [ ]:
y_pred = model.predict(X_test_sc)
probs  = model.predict_proba(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#4878CF', linewidth=2,
             label=f'Naïve Bayes (AUC={roc_auc:.3f})')
axes[1].plot([0,1],[0,1],'gray',linestyle='--',linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()
print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 7. Effect of the Independence Assumption

Let's check how correlated our features actually are — to see how badly we're violating the naïve assumption.

In [ ]:
corr_matrix = pd.DataFrame(X_train_sc, columns=feature_cols).corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix\n(Naïve Bayes assumes all off-diagonal values = 0)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

Non-zero off-diagonal values show feature correlations — exactly what Naïve Bayes ignores. Despite this, the model still performs reasonably. The independence assumption is violated but not catastrophically so for this dataset.

---
## 8. Discussion — Does Naïve Bayes Suit Football?

### What it does well
- **Instant training**: no gradient descent — one pass through data is all it takes
- **Works with small data**: estimates only 2 parameters per feature per class — very few parameters to learn
- **Probabilistic output**: clean probability interpretation via Bayes' theorem
- **Interpretable**: the learned Gaussians directly show how each feature differs between classes
- **Scales well**: prediction is just evaluating Gaussian PDFs — very fast

### What it struggles with
1. **Independence assumption violated**: football features ARE correlated — home goals and home win rate move together. This hurts calibration.
2. **Gaussian assumption**: some features may not be normally distributed within each class
3. **Can't capture interactions**: by design, the model treats each feature in isolation
4. **Football noise**: the model's simplicity means it may underfit the complex patterns in match outcomes

### Where Naïve Bayes truly shines

Spam filtering, text classification, medical diagnosis with independent test results — domains where features are genuinely (or approximately) independent and data may be limited. For football, more complex models generally outperform it, but it provides a fast and interpretable baseline.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Probabilistic generative classifier |
| **Approach** | Bayes' theorem + independence assumption |
| **Training** | Compute means & variances per class — instant |
| **Assumption** | Features are conditionally independent given class |
| **Output** | Calibrated probability via posterior |
| **Needs scaling** | Not strictly, but helps numerical stability |
| **Key concept introduced** | Bayes' theorem, generative modeling, log-sum-exp trick |
| **Football suitability** | Moderate — fast baseline, but independence assumption hurts |
